# Analyze saved results: trajectories, RUL, posterior distributions

Loads what `scripts/full_run.py` (full-data run) and `scripts/esticol_sweep.py` (ESTICOL
observation-size sweep) already saved to disk -- no assimilation is (re)run here, this
notebook is purely a viewer/plotter over:

- `results/trajectories/<SG>_<stage>.npy` (+ `_time.npy`) and
  `results/posteriors/<SG>_cycle<i>_posterior.csv` -- the full, all-ESTICOL run
- `results_sweep/<SG>/trajectories/<SG>_k<k>_<stage>.npy` (+ `_time.npy`) and
  `results_sweep/<SG>/posteriors/<SG>_k<k>_cycle<i>_posterior.csv` -- one set per `k`
  in the ESTICOL sweep

Sections: (1) load & discover what's on disk, (2) the prior/BMU/BMU+EnKS trajectory +
RUL-histogram grid for one run (full or a given `k`), (3) RUL vs `k` across the whole
sweep, (4) posterior-parameter distributions per cycle, (5) how the active variable's
posterior sharpens as `k` grows.

In [ ]:
import os
import sys

# Run from the application root so that DATA/, results*/ and figures/ resolve, and so that
# the application package is importable (pip install -e ../.. makes bayes_fusion importable).
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())
os.makedirs('figures/preprocessing', exist_ok=True)
os.makedirs('figures/analysis', exist_ok=True)

In [ ]:
import glob
import os
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import gaussian_kde, kstest

from clogging_prognostics import PipelineConfig, first_crossing_rul
from clogging_prognostics.data_preparation import preprocess_field_data

RESULTS_DIR = 'results'                              # scripts/full_run.py output
SWEEP_DIR = 'results_sweep'                          # scripts/esticol_sweep.py output (both SGs)
BASE_DIRECTORY = 'DATA/'
SG_TAGS = ['SGA', 'SGB']                             # file prefixes in DATA/ and results*/
SG_DISPLAY = {'SGA': 'SG A', 'SGB': 'SG B'}          # names shown in plots and tables
STAGE_ORDER = ['prior', 'bmu', 'bmu_enks']
STAGE_LABEL = {'prior': 'Prior', 'bmu': 'BMU', 'bmu_enks': 'BMU+EnKS'}
STAGE_COLOR = {'prior': 'green', 'bmu': 'royalblue', 'bmu_enks': 'lightcoral'}
DEFAULT_THRESHOLD = None

# Reference prior (names + bounds) used to show the un-informed marginal
# alongside each posterior, and to detect which variable was HSIC-active in
# a given cycle -- purely for display, no assimilation happens here.
_PRIOR = PipelineConfig(base_directory=BASE_DIRECTORY).prior
PARAM_NAMES = list(_PRIOR.names)
PARAM_BOUNDS = dict(zip(_PRIOR.names, _PRIOR.bounds))
plt.rcParams.update({'font.size': 12, 'figure.figsize': (6, 4), 'figure.dpi': 300, 'text.usetex': True})

## Loaders

In [ ]:
def _traj_dir(sg_tag, k):
    return os.path.join(RESULTS_DIR, 'trajectories') if k is None \
        else os.path.join(SWEEP_DIR, 'trajectories')


def _posteriors_dir(sg_tag, k):
    return os.path.join(RESULTS_DIR, 'posteriors') if k is None \
        else os.path.join(SWEEP_DIR, 'posteriors')


def _prefix(sg_tag, k):
    return sg_tag if k is None else f'{sg_tag}_k{k:04d}'


def load_trajectories(sg_tag, k=None):
    """Load {stage: (traj, time)} for the full run (k=None) or one sweep k."""
    d, p = _traj_dir(sg_tag, k), _prefix(sg_tag, k)
    return {
        stage: (np.load(os.path.join(d, f'{p}_{stage}.npy')),
                np.load(os.path.join(d, f'{p}_{stage}_time.npy')))
        for stage in STAGE_ORDER
    }


def load_posterior(sg_tag, cycle_index, k=None):
    """Load one cycle's posterior-parameter-sample DataFrame."""
    d, p = _posteriors_dir(sg_tag, k), _prefix(sg_tag, k)
    return pd.read_csv(os.path.join(d, f'{p}_cycle{cycle_index}_posterior.csv'))


def discover_sweep_ks(sg_tag):
    """Which k values have saved trajectories for this SG's ESTICOL sweep."""
    d = _traj_dir(sg_tag, k=0)                        # any k -> same directory
    if not os.path.isdir(d):
        return []
    ks = set()
    for f in glob.glob(os.path.join(d, f'{sg_tag}_k*_prior.npy')):
        m = re.search(rf'{re.escape(sg_tag)}_k(\d+)_prior\.npy$', os.path.basename(f))
        if m:
            ks.add(int(m.group(1)))
    return sorted(ks)


def discover_cycles(sg_tag, k=None):
    """Which cycle indices have a saved posterior for this run/k."""
    d, p = _posteriors_dir(sg_tag, k), _prefix(sg_tag, k)
    if not os.path.isdir(d):
        return []
    cycles = set()
    for f in glob.glob(os.path.join(d, f'{p}_cycle*_posterior.csv')):
        m = re.search(rf'{re.escape(p)}_cycle(\d+)_posterior\.csv$', os.path.basename(f))
        if m:
            cycles.add(int(m.group(1)))
    return sorted(cycles)


def has_full_run(sg_tag):
    return bool(discover_cycles(sg_tag, k=None))


def load_last_cycle_field_data(sg_tag):
    """Raw ESTICOL/ETV points for this SG's last (swept) cycle.

    Splitting into first-k-used / held-out subsets is left to the caller.
    """
    import json as _json
    meta = _json.load(open(os.path.join(BASE_DIRECTORY, f'{sg_tag}_METADATA.json')))
    sim_time = np.load(os.path.join(BASE_DIRECTORY, f'{sg_tag}_TIME.npy'), allow_pickle=True)
    cleaning_dates = [c['time'] / 24 for c in meta['chemistry']['cleaning']]
    data_by_cycle, _ = preprocess_field_data(
        csv_file=os.path.join(BASE_DIRECTORY, f'{sg_tag}_FIELD_DATA.csv'),
        operation_time=sim_time, cleaning_dates=cleaning_dates, with_time_division=True,
    )
    esticol, etv = data_by_cycle[-1]                  # last cycle -- the one the sweep truncates
    return np.asarray(esticol), np.asarray(etv)

## What's on disk

In [ ]:
for sg_tag in SG_TAGS:
    full = 'yes' if has_full_run(sg_tag) else 'no'
    ks = discover_sweep_ks(sg_tag)
    print(f'{sg_tag}: full run saved = {full} | sweep k values saved = {ks}')

## Trajectories + RUL for one run (full data, or a single sweep `k`)

Same 2x3 layout as in `two_sg_fusion_methodology.ipynb` (top: cumulative RUL
histogram+KDE per stage; bottom: that stage's own trajectory ensemble), rebuilt here
purely from the saved `.npy` files. For a sweep `k`, the last cycle's ESTICOL points are
split into *used* (first `k`, chronologically) and *held out* (the rest), so the plot
shows exactly what informed that run.

In [ ]:
def plot_prognostics_summary(sg_tag, k=None, threshold=DEFAULT_THRESHOLD, title=None):
    trajectories = load_trajectories(sg_tag, k=k)
    esticol, etv = load_last_cycle_field_data(sg_tag)

    if k is None:
        esticol_used, esticol_held = esticol, np.zeros((0, 2))
    else:
        order = np.argsort(esticol[:, 0]) if len(esticol) else np.array([], dtype=int)
        esticol_sorted = esticol[order]
        kk = min(k, len(esticol_sorted))
        esticol_used, esticol_held = esticol_sorted[:kk], esticol_sorted[kk:]

    rul_by_stage = {}
    for stage in STAGE_ORDER:
        traj, t = trajectories[stage]
        rul_by_stage[stage] = first_crossing_rul(traj, t, threshold)

    fig, axes = plt.subplots(2, 3, figsize=(12, 7))
    for col, stage in enumerate(STAGE_ORDER):
        traj, t = trajectories[stage]

        ax = axes[0, col]
        for s2 in STAGE_ORDER[:col + 1]:
            rul = rul_by_stage[s2]
            if rul.size == 0:
                continue
            counts, bins, _ = ax.hist(rul, bins=10, color=STAGE_COLOR[s2], alpha=0.9,
                                      edgecolor='k', label=f'{STAGE_LABEL[s2]} RUL')
            if rul.size > 1 and np.std(rul) > 0:
                kde = gaussian_kde(rul)
                xs = np.linspace(bins[0], bins[-1], 200)
                ax.plot(xs, kde(xs) * rul.size * (bins[1] - bins[0]), color=STAGE_COLOR[s2])
        ax.set_ylabel('Frequency')
        if col > 0:
            ax.legend(fontsize=8)

        axb = axes[1, col]
        axb.axvspan(t.min(), t.max(), color='lightgray', zorder=0)
        for i in range(traj.shape[0]):
            axb.plot(t, traj[i], color=STAGE_COLOR[stage], alpha=0.15, linewidth=0.8)
        axb.axhline(threshold, color='darkred', linestyle='--', linewidth=1.5,
                    label=r'$\tau_c^\ast$' if col == 0 else None)
        if len(esticol_used):
            axb.scatter(esticol_used[:, 0], esticol_used[:, 1], c='red', marker='o',
                        edgecolors='k', linewidths=0.4, s=50, zorder=11,
                        label='ESTICOL' if col == 0 else None)
        if len(esticol_held):
            axb.scatter(esticol_held[:, 0], esticol_held[:, 1], c='red', marker='o',
                    edgecolors='k', linewidths=0.4, s=50, zorder=11, alpha=0.25,
                        label='ESTICOL (held out)' if col == 0 else None)
        if len(etv):
            axb.scatter(etv[:, 0], etv[:, 1], c='blue', marker='^',
                        edgecolors='k', linewidths=0.4, s=60, zorder=12,
                        label='TVE' if col == 0 else None)
        axb.set_xlabel(r'$t\;$ (days)', fontsize=20)
        axb.set_xlim(t.min(), t.max())
        if col == 0:
            axb.set_ylabel(r'$\tau_c$', fontsize=20)
            axb.legend(fontsize=8, loc='lower right')
        axb.set_title(STAGE_LABEL[stage])
        axb.set_ylim(0,100)
        axb.set_yticks([])

    fig.suptitle(title or (f'{SG_DISPLAY[sg_tag]} - full run' if k is None else f'{SG_DISPLAY[sg_tag]} - k={k}'))
    fig.tight_layout()
    return fig

In [ ]:
THRESHOLDS = {'SGA': None, 'SGB': None}          # tau_c^* (%) per steam generator

In [ ]:
# Example: full-data run, both SGs (skips a SG silently if not yet saved)
for sg_tag in SG_TAGS:
    if has_full_run(sg_tag):
        plot_prognostics_summary(sg_tag, threshold=THRESHOLDS[sg_tag], k=None)
        plt.savefig(f'figures/analysis/{sg_tag}_prognostics_summary_full_run.pdf', dpi=300, bbox_inches='tight', format='pdf')
        plt.show()

    else:
        print(f'{sg_tag}: no full run saved yet -- run scripts/full_run.py first')

In [ ]:
# Example: one sweep k (edit SG_TAG / K to taste)
SG_TAG, K = 'SGB', 20
if K in discover_sweep_ks(SG_TAG):
    plot_prognostics_summary(SG_TAG, k=K, threshold=THRESHOLDS[SG_TAG])
    plt.savefig(f'figures/analysis/{SG_TAG}_prognostics_summary_k{K:04d}.pdf', dpi=300, bbox_inches='tight', format='pdf')
    plt.show()
else:
    print(f'{SG_TAG}: k={K} not found, available: {discover_sweep_ks(SG_TAG)}')

## RUL vs. ESTICOL count `k`

Rebuilds the mean-RUL-vs-`k` curve directly from the saved trajectories (via
`first_crossing_rul`), one curve per stage, with an IQR band -- the key figure for the
"value of ESTICOL" question: does the RUL estimate keep sharpening as more field data is
assimilated, and where does it plateau?

In [ ]:
def rul_vs_k_dataframe(sg_tag, threshold=DEFAULT_THRESHOLD):
    """Per-(k, stage) RUL statistics; same stat columns as rul_stats_table."""
    rows = []
    for k in discover_sweep_ks(sg_tag):
        trajectories = load_trajectories(sg_tag, k=k)
        for stage in STAGE_ORDER:
            traj, t = trajectories[stage]
            rul = first_crossing_rul(traj, t, threshold)
            if rul.size == 0:
                rows.append(dict(k=k, stage=stage, threshold=threshold,
                                 count=0, mean=np.nan, std=np.nan, min=np.nan,
                                 q25=np.nan, median=np.nan, q75=np.nan, max=np.nan,
                                 n_total=traj.shape[0]))
            else:
                rows.append(dict(k=k, stage=stage, threshold=threshold,
                                 count=rul.size, mean=rul.mean(), std=rul.std(),
                                 min=rul.min(), q25=np.quantile(rul, 0.25),
                                 median=np.median(rul), q75=np.quantile(rul, 0.75),
                                 max=rul.max(), n_total=traj.shape[0]))
    return pd.DataFrame(rows)


def plot_rul_vs_k(figsize=(9, 3.5)):
    """Mean RUL + IQR vs. k, one subplot per SG (own threshold each)."""
    fig, axes = plt.subplots(1, 2, figsize=figsize, sharey=False)
    dfs = {}
    for ax, sg_tag in zip(axes, SG_TAGS):
        threshold = THRESHOLDS[sg_tag]
        df = rul_vs_k_dataframe(sg_tag, threshold)
        dfs[sg_tag] = df
        for stage in STAGE_ORDER:
            d = df[df['stage'] == stage].sort_values('k')
            ax.plot(d['k'], d['mean'], '-o', color=STAGE_COLOR[stage], label=STAGE_LABEL[stage],
                    markersize=4, linewidth=1.5)
            ax.fill_between(d['k'], d['q25'], d['q75'], color=STAGE_COLOR[stage], alpha=0.2)
        ax.set_xlabel('$k$', fontsize=9)
        ax.set_ylabel('RUL (days)', fontsize=9)
        ax.set_title(SG_DISPLAY[sg_tag], fontsize=10)
        ax.tick_params(labelsize=7)
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=6)
    fig.suptitle('RUL mean + IQR vs. ESTICOL count', fontsize=11)
    fig.tight_layout()
    return fig, dfs

In [ ]:
if any(len(discover_sweep_ks(sg_tag)) >= 2 for sg_tag in SG_TAGS):
    fig, dfs = plot_rul_vs_k()
    plt.savefig('figures/analysis/rul_vs_k_summary.pdf', dpi=300, bbox_inches='tight', format='pdf')
    plt.show()
    for sg_tag in SG_TAGS:
        print(sg_tag)
        display(dfs[sg_tag].round(1))
else:
    print('need >= 2 saved k values for at least one SG to plot a trend')

## RUL histograms overlaid across `k`

For one stage, overlay the RUL distribution at every saved `k` -- shows the
distribution sharpening/shifting directly, rather than only its mean+IQR.

In [ ]:
def plot_rul_histograms_vs_k(stage='bmu_enks', figsize=(9, 3.5)):
    """Overlay RUL KDE density curves vs. k, one subplot per SG (own threshold each)."""
    fig, axes = plt.subplots(1, 2, figsize=figsize, sharey=False)
    for ax, sg_tag in zip(axes, SG_TAGS):
        threshold = THRESHOLDS[sg_tag]
        ks = discover_sweep_ks(sg_tag)
        cmap = plt.cm.viridis(np.linspace(0, 1, max(len(ks), 1)))
        for color, k in zip(cmap, ks):
            traj, t = load_trajectories(sg_tag, k=k)[stage]
            rul = first_crossing_rul(traj, t, threshold)
            if rul.size < 2 or np.std(rul) == 0:
                continue
            kde = gaussian_kde(rul)
            xs = np.linspace(rul.min(), rul.max(), 200)
            ax.plot(xs, kde(xs), color=color, linewidth=1.8, label=f'k={k}')
            ax.fill_between(xs, kde(xs), color=color, alpha=0.08)
        ax.set_xlabel('RUL (days)', fontsize=9)
        ax.set_ylabel('Density', fontsize=9)
        ax.set_title(f'{SG_DISPLAY[sg_tag]}: {STAGE_LABEL[stage]}', fontsize=10)
        ax.tick_params(labelsize=7)
        ax.legend(fontsize=6)
        ax.set_ylim(0,)
    fig.tight_layout()
    return fig

In [ ]:
if any(len(discover_sweep_ks(sg_tag)) >= 2 for sg_tag in SG_TAGS):
    plot_rul_histograms_vs_k(stage='bmu_enks')
    plt.savefig('figures/analysis/rul_histograms_vs_k_summary.pdf', dpi=300, bbox_inches='tight', format='pdf')
    plt.show()

## Posterior-parameter distributions per cycle

One histogram per parameter per cycle, with the prior's uniform range shaded behind it,
for context. The HSIC-active variable in a given cycle is the one whose posterior looks
least like its (uniform) prior -- flagged via a Kolmogorov-Smirnov test against that
reference, purely for display (no cycle-diagnostics file is read here).

In [ ]:
def _is_likely_active(samples, lo, hi):
    if hi <= lo or samples.std() == 0:
        return False
    stat, _ = kstest(samples, 'uniform', args=(lo, hi - lo))
    return stat > 0.15                # heuristic threshold, purely for the (*) marker

PARAM_NAMES_ = [r"$\alpha$", r"$\beta$", r"$\epsilon_e$", r"$\epsilon_c$", r"$d_p$", r"$\Gamma_{p}(0)$", r"$a_v$"]
COLORS_ = ['royalblue', 'orange', 'green']

def plot_posterior_grid(sg_tag, k=None, cycles=None):
    cycles = cycles if cycles is not None else discover_cycles(sg_tag, k=k)
    if not cycles:
        print(f'{sg_tag}: no posteriors saved for k={k}')
        return None

    fig, axes = plt.subplots(len(cycles), len(PARAM_NAMES),
                             figsize=(2.2 * len(PARAM_NAMES), 2.2 * len(cycles)),
                             squeeze=False)
    
    for row, cycle_index in enumerate(cycles):
        df = load_posterior(sg_tag, cycle_index, k=k)
    
        for col, name in enumerate(PARAM_NAMES):
            ax = axes[row, col]
            lo, hi = PARAM_BOUNDS[name]
            samples = df[name].to_numpy()
            ax.hist(samples, bins=15, color=COLORS_[row], edgecolor='k', alpha=0.8, zorder=1)
            active = _is_likely_active(samples, lo, hi)
            if active:
                suffix = '*'
            else:
                suffix = ''
            if row == 0:
                ax.set_title(f'{PARAM_NAMES_[col]}{suffix}', fontsize=20)
            if col == 0:
                ax.set_ylabel(f'cycle {cycle_index}', fontsize=20)
            ax.tick_params(labelsize=7)
            ax.set_xlim(lo, hi)
            ax.set_yticks([])

    title = f'{SG_DISPLAY[sg_tag]} posteriors ' + ('full run' if k is None else f'k={k}')
    fig.suptitle(title, fontsize=20)
    fig.tight_layout()
    return fig

In [ ]:
for sg_tag in SG_TAGS:
    if has_full_run(sg_tag):
        plot_posterior_grid(sg_tag, k=None)
        plt.savefig(f'figures/analysis/{sg_tag}_posterior_grid_full_run.pdf', dpi=300, bbox_inches='tight', format='pdf')
        plt.show()

## Active variable's posterior vs. `k`

For the swept (last) prognostic window only, overlay the posterior of a chosen parameter
(here `a_v`) at every saved `k`, one subplot per SG -- the direct visual answer to "does more
ESTICOL data narrow the calibrated parameter's uncertainty?". Earlier cycles are left out: their
ESTICOL points do not change with `k`, so they carry no sensitivity information.

In [ ]:
def _sci_tick(v, _pos=None):
    if v == 0:
        return r'$0$'
    e = int(np.floor(np.log10(abs(v))))
    return rf'${v / 10**e:.1f}\times10^{{{e}}}$'


def plot_active_posterior_vs_k(param=None, figsize=(7, 3.2)):
    """Shaded KDEs of `param`'s posterior on the last (swept) cycle, one curve per k, one subplot per SG.

    Only the last cycle is shown: earlier cycles see the same ESTICOL points for every k, so
    they carry no sensitivity to k. param=None picks each SG's active variable (largest KS
    distance from its uniform prior) from its largest-k run. The x-range is the prior range
    (extended if samples fall outside it) and the curves fill it edge to edge.
    """
    from matplotlib.ticker import FuncFormatter, MaxNLocator

    fig, axes = plt.subplots(1, len(SG_TAGS), figsize=figsize, squeeze=False)
    for ax, sg_tag in zip(axes[0], SG_TAGS):
        ks = discover_sweep_ks(sg_tag)
        if not ks:
            ax.set_title(f'{SG_DISPLAY[sg_tag]}: no sweep saved yet', fontsize=10)
            continue
        last_cycle = max(discover_cycles(sg_tag, k=ks[-1]))
        samples_by_k = {k: load_posterior(sg_tag, last_cycle, k=k) for k in ks}

        name = param
        if name is None:
            df_max = samples_by_k[ks[-1]]
            scores = {n: kstest(df_max[n], 'uniform',
                                args=(PARAM_BOUNDS[n][0],
                                      PARAM_BOUNDS[n][1] - PARAM_BOUNDS[n][0]))[0]
                      for n in PARAM_NAMES}
            name = max(scores, key=scores.get)
        label = PARAM_NAMES_[PARAM_NAMES.index(name)]

        lo = min([PARAM_BOUNDS[name][0]] + [df[name].min() for df in samples_by_k.values()])
        hi = max([PARAM_BOUNDS[name][1]] + [df[name].max() for df in samples_by_k.values()])
        xs = np.linspace(lo, hi, 300)
        cmap = plt.cm.viridis(np.linspace(0, 1, len(ks)))
        for color, k in zip(cmap, ks):
            s = samples_by_k[k][name].to_numpy()
            if s.size < 2 or np.std(s) == 0:
                continue
            dens = gaussian_kde(s)(xs)
            ax.plot(xs, dens, color=color, linewidth=1.8, label=f'k={k}')
            ax.fill_between(xs, dens, color=color, alpha=0.08)
        ax.set_xlabel(label, fontsize=15)
        ax.set_title(f'{SG_DISPLAY[sg_tag]} (cycle {last_cycle})', fontsize=10)
        ax.tick_params(labelsize=7)
        ax.xaxis.set_major_locator(MaxNLocator(nbins=3))
        ax.xaxis.set_major_formatter(FuncFormatter(_sci_tick))
        ax.set_xlim(lo, hi)
        ax.set_ylim(0,)
        ax.margins(x=0)
        ax.set_yticks([])
        ax.legend(fontsize=10)
    axes[0, 0].set_ylabel('Density', fontsize=9)
    fig.tight_layout()
    return fig

In [ ]:
if any(len(discover_sweep_ks(sg_tag)) >= 2 for sg_tag in SG_TAGS):
    plot_active_posterior_vs_k(param='a_v')
    plt.savefig('figures/analysis/a_v_posterior_vs_k.pdf', dpi=300, bbox_inches='tight', format='pdf')
    plt.show()

## Summary tables: RUL statistics and ESTICOL sweeps

Tabular view of the same numbers behind the plots above -- per-SG RUL summary
statistics for the full run, and the RUL-vs-`k` sweep table, each computed with
that SG's own threshold (from `THRESHOLDS` in the
"What's on disk" section).

In [ ]:
def rul_stats_table(sg_tag, threshold):
    """Summary statistics (count/mean/std/quantiles) of RUL per stage, full run."""
    trajectories = load_trajectories(sg_tag, k=None)
    rows = []
    for stage in STAGE_ORDER:
        traj, t = trajectories[stage]
        rul = first_crossing_rul(traj, t, threshold)
        if rul.size == 0:
            rows.append(dict(SG=SG_DISPLAY[sg_tag], stage=STAGE_LABEL[stage], threshold=threshold,
                             count=0, mean=np.nan, std=np.nan, min=np.nan,
                             q25=np.nan, median=np.nan, q75=np.nan, max=np.nan))
        else:
            rows.append(dict(SG=SG_DISPLAY[sg_tag], stage=STAGE_LABEL[stage], threshold=threshold,
                             count=rul.size, mean=rul.mean(), std=rul.std(),
                             min=rul.min(), q25=np.quantile(rul, 0.25),
                             median=np.median(rul), q75=np.quantile(rul, 0.75),
                             max=rul.max()))
    return pd.DataFrame(rows)


rul_stats_frames = []
for sg_tag in SG_TAGS:
    if has_full_run(sg_tag):
        rul_stats_frames.append(rul_stats_table(sg_tag, THRESHOLDS[sg_tag]))
    else:
        print(f'{sg_tag}: no full run saved yet -- skipping RUL stats table')

if rul_stats_frames:
    rul_stats_df = pd.concat(rul_stats_frames, ignore_index=True)
    display(rul_stats_df.round(2))

In [ ]:
sweep_frames = []
for sg_tag in SG_TAGS:
    ks = discover_sweep_ks(sg_tag)
    if not ks:
        print(f'{sg_tag}: no sweep k values saved -- skipping sweep table')
        continue
    df = rul_vs_k_dataframe(sg_tag, threshold=THRESHOLDS[sg_tag])
    df['stage'] = df['stage'].map(STAGE_LABEL)
    df.insert(0, 'SG', SG_DISPLAY[sg_tag])
    sweep_frames.append(df)

if sweep_frames:
    sweep_df = pd.concat(sweep_frames, ignore_index=True)
    display(sweep_df.round(2))

## ESTICOL kinetics experiment (`results_kinetics/`)

Loads what `scripts/esticol_kinetics.py` saved for the SG B "kinetics" experiment -- instead of a
first-`k` sweep, each run assimilates a chosen *sub-population* of the last cycle's ESTICOL
points (TVE data held fixed):

- `quasi_linear`: the first 20 ESTICOL points after the last chemical cleaning
- `exponential`: the last 20 ESTICOL points, counting backwards from the last one

Layout: `results_kinetics/trajectories/<SG>_<experiment>_<stage>.npy` (+ `_time.npy`),
`results_kinetics/posteriors/<SG>_<experiment>_cycle<i>_posterior.csv` and
`results_kinetics/esticol_points/<SG>_<experiment>_esticol_points.csv` (the points actually
assimilated). Same 2x3 plot + RUL-statistics-table pattern as the sections above.

In [ ]:
KINETICS_DIR = 'results_kinetics'                    # scripts/esticol_kinetics.py output


def discover_kinetics_experiments(sg_tag):
    """Which experiment names have saved trajectories for this SG."""
    d = os.path.join(KINETICS_DIR, 'trajectories')
    if not os.path.isdir(d):
        return []
    names = set()
    for f in glob.glob(os.path.join(d, f'{sg_tag}_*_prior.npy')):
        m = re.search(rf'{re.escape(sg_tag)}_(.+)_prior\.npy$', os.path.basename(f))
        if m:
            names.add(m.group(1))
    return sorted(names)


def load_kinetics_trajectories(sg_tag, name):
    """Load {stage: (traj, time)} for one kinetics experiment."""
    d, p = os.path.join(KINETICS_DIR, 'trajectories'), f'{sg_tag}_{name}'
    return {
        stage: (np.load(os.path.join(d, f'{p}_{stage}.npy')),
                np.load(os.path.join(d, f'{p}_{stage}_time.npy')))
        for stage in STAGE_ORDER
    }


def discover_kinetics_cycles(sg_tag, name):
    """Which cycle indices have a saved posterior for this SG/experiment."""
    d, p = os.path.join(KINETICS_DIR, 'posteriors'), f'{sg_tag}_{name}'
    if not os.path.isdir(d):
        return []
    cycles = set()
    for f in glob.glob(os.path.join(d, f'{p}_cycle*_posterior.csv')):
        m = re.search(rf'{re.escape(p)}_cycle(\d+)_posterior\.csv$', os.path.basename(f))
        if m:
            cycles.add(int(m.group(1)))
    return sorted(cycles)


def load_kinetics_posterior(sg_tag, name, cycle_index):
    """Load one cycle's posterior-parameter-sample DataFrame for a kinetics experiment."""
    d, p = os.path.join(KINETICS_DIR, 'posteriors'), f'{sg_tag}_{name}'
    return pd.read_csv(os.path.join(d, f'{p}_cycle{cycle_index}_posterior.csv'))


def load_kinetics_esticol_points(sg_tag, name):
    """The (time, tau_c) ESTICOL points this experiment actually assimilated."""
    path = os.path.join(KINETICS_DIR, 'esticol_points', f'{sg_tag}_{name}_esticol_points.csv')
    return pd.read_csv(path).to_numpy()

In [ ]:
for sg_tag in SG_TAGS:
    exps = discover_kinetics_experiments(sg_tag)
    print(f'{sg_tag}: kinetics experiments saved = {exps}')

### Trajectories + RUL per kinetics experiment

Same 2x3 layout as the full-run/sweep section above, but the ESTICOL points scattered on the
bottom row are split into *used* (the exact sub-population this experiment assimilated, from
`esticol_points/`) and *held out* (every other last-cycle ESTICOL point, for context) --
`quasi_linear` and `exponential` do not use a first-`k` split, so this is not the same as the
sweep's used/held-out.

In [ ]:
def plot_kinetics_summary(sg_tag, name, threshold=DEFAULT_THRESHOLD, title=None):
    trajectories = load_kinetics_trajectories(sg_tag, name)
    esticol_all, etv = load_last_cycle_field_data(sg_tag)
    esticol_used = load_kinetics_esticol_points(sg_tag, name)

    used_times = set(np.round(esticol_used[:, 0], 6)) if len(esticol_used) else set()
    held_mask = np.array([round(t, 6) not in used_times for t in esticol_all[:, 0]]) \
        if len(esticol_all) else np.zeros(0, dtype=bool)
    esticol_held = esticol_all[held_mask] if len(esticol_all) else esticol_all

    rul_by_stage = {}
    for stage in STAGE_ORDER:
        traj, t = trajectories[stage]
        rul_by_stage[stage] = first_crossing_rul(traj, t, threshold)

    fig, axes = plt.subplots(2, 3, figsize=(12, 7))
    for col, stage in enumerate(STAGE_ORDER):
        traj, t = trajectories[stage]

        ax = axes[0, col]
        for s2 in STAGE_ORDER[:col + 1]:
            rul = rul_by_stage[s2]
            if rul.size == 0:
                continue
            counts, bins, _ = ax.hist(rul, bins=10, color=STAGE_COLOR[s2], alpha=0.9,
                                      edgecolor='k', label=f'{STAGE_LABEL[s2]} RUL')
            if rul.size > 1 and np.std(rul) > 0:
                kde = gaussian_kde(rul)
                xs = np.linspace(bins[0], bins[-1], 200)
                ax.plot(xs, kde(xs) * rul.size * (bins[1] - bins[0]), color=STAGE_COLOR[s2])
        ax.set_ylabel('Frequency')
        if col > 0:
            ax.legend(fontsize=8)

        axb = axes[1, col]
        axb.axvspan(t.min(), t.max(), color='lightgray', zorder=0)
        for i in range(traj.shape[0]):
            axb.plot(t, traj[i], color=STAGE_COLOR[stage], alpha=0.15, linewidth=0.8)
        axb.axhline(threshold, color='darkred', linestyle='--', linewidth=1.5,
                    label=r'$\tau_c^\ast$' if col == 0 else None)
        if len(esticol_used):
            axb.scatter(esticol_used[:, 0], esticol_used[:, 1], c='red', marker='o',
                        edgecolors='k', linewidths=0.4, s=50, zorder=11,
                        label='ESTICOL (used)' if col == 0 else None)
        if len(esticol_held):
            axb.scatter(esticol_held[:, 0], esticol_held[:, 1], c='red', marker='o',
                    edgecolors='k', linewidths=0.4, s=50, zorder=11, alpha=0.25,
                        label='ESTICOL (held out)' if col == 0 else None)
        if len(etv):
            axb.scatter(etv[:, 0], etv[:, 1], c='blue', marker='^',
                        edgecolors='k', linewidths=0.4, s=60, zorder=12,
                        label='TVE' if col == 0 else None)
        axb.set_xlabel(r'$t\;$ (days)', fontsize=20)
        axb.set_xlim(t.min(), t.max())
        if col == 0:
            axb.set_ylabel(r'$\tau_c$', fontsize=20)
            axb.legend(fontsize=8, loc='lower right')
        axb.set_title(STAGE_LABEL[stage])
        axb.set_ylim(0, 100)
        axb.set_yticks([])

    fig.suptitle(title or f'{SG_DISPLAY[sg_tag]} - {name}')
    fig.tight_layout()
    return fig

In [ ]:
for sg_tag in SG_TAGS:
    for name in discover_kinetics_experiments(sg_tag):
        plot_kinetics_summary(sg_tag, name, threshold=THRESHOLDS[sg_tag])
        plt.savefig(f'figures/analysis/{sg_tag}_{name}_kinetics_summary.pdf', dpi=300, bbox_inches='tight', format='pdf')
        plt.show()

### RUL statistics per kinetics experiment

Same stat columns as `rul_stats_table`/`rul_vs_k_dataframe` above, one row per (SG, experiment,
stage), plus `n_points`: the number of ESTICOL points that experiment actually assimilated.

In [ ]:
def kinetics_stats_table(sg_tag, name, threshold):
    """Summary statistics (count/mean/std/quantiles) of RUL per stage, one kinetics experiment."""
    trajectories = load_kinetics_trajectories(sg_tag, name)
    n_points = len(load_kinetics_esticol_points(sg_tag, name))
    rows = []
    for stage in STAGE_ORDER:
        traj, t = trajectories[stage]
        rul = first_crossing_rul(traj, t, threshold)
        if rul.size == 0:
            rows.append(dict(SG=SG_DISPLAY[sg_tag], experiment=name, n_points=n_points,
                             stage=STAGE_LABEL[stage], threshold=threshold,
                             count=0, mean=np.nan, std=np.nan, min=np.nan,
                             q25=np.nan, median=np.nan, q75=np.nan, max=np.nan))
        else:
            rows.append(dict(SG=SG_DISPLAY[sg_tag], experiment=name, n_points=n_points,
                             stage=STAGE_LABEL[stage], threshold=threshold,
                             count=rul.size, mean=rul.mean(), std=rul.std(),
                             min=rul.min(), q25=np.quantile(rul, 0.25),
                             median=np.median(rul), q75=np.quantile(rul, 0.75),
                             max=rul.max()))
    return pd.DataFrame(rows)


kinetics_frames = []
for sg_tag in SG_TAGS:
    for name in discover_kinetics_experiments(sg_tag):
        kinetics_frames.append(kinetics_stats_table(sg_tag, name, THRESHOLDS[sg_tag]))

if kinetics_frames:
    kinetics_df = pd.concat(kinetics_frames, ignore_index=True)
    display(kinetics_df.round(2))
else:
    print('no results_kinetics/ outputs found yet -- run scripts/esticol_kinetics.py first')